<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/CAM_DS_C301_Implementing_advanced_RNNs_Demo_2_1_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<br>

**First things first** - please go to 'File' and select 'Save a copy in Drive' so that you have your own version of this activity set up and ready to use.
Remember to update the portfolio index link to your own work once completed!

# Demonstration 2.1.4 Implementing advanced RNNs

In this demonstration, we will explore advanced recurrent neural network architectures, specifically long short-term memory networks (LSTMs) and gated recurrent units (GRUs), using the IMDb data set. We will begin by running a simple RNN for benchmarking purposes. Following this, we will implement an LSTM network by incorporating an LSTM layer into our existing model and observe the significant improvements in performance and accuracy. Finally, we will replace the LSTM layer with a GRU layer to compare the architectures, noting the similarities in accuracy and their superiority over a simple RNN, thanks to their advanced gating mechanisms that address the vanishing gradient problem.


In [ ]:
!pip install datasets

## Importing necessary libraries

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM,SimpleRNN, GRU, Bidirectional,SpatialDropout1D
from tensorflow.keras.datasets import imdb


##  Loading the data set

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Create dataframes of the train and validation split.
text_train = dataset['train']['text']
label_train = dataset['train']['label']
text_test = dataset['test']['text']
label_test = dataset['test']['label']

In [ ]:
import pandas as pd
df_train = pd.DataFrame()
df_train['text'] = text_train
df_train['label'] = label_train

In [ ]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [ ]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(df_train,  test_size=0.2)

In [ ]:
text_train =  train['text'].tolist()
label_train = train['label'].tolist()
text_valid = valid['text'].tolist()
label_valid = valid['label'].tolist()


## Preprocessing the data
Sequences of numbers (words) are of different lengths. We need to pad them so that they have the same length for modelling.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Parameters.
vocab_size = 10000
max_length = 150
padding_type = 'post'
trunc_type = 'post'

# Initialise the tokeniser.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(dataset['train']['text'])

# Tokenise the sentences and pad the sequences in the training set.
train_sequences = tokenizer.texts_to_sequences(text_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)


# Tokenize the sentences and pad the sequences in the validation set.
valid_sequences = tokenizer.texts_to_sequences(text_valid)
valid_padded = pad_sequences(valid_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Tokenize the sentences and pad the sequences in the test set.
test_sequences = tokenizer.texts_to_sequences(text_test )
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Prepare the labels.
train_labels = np.array(label_train)
valid_labels = np.array(label_valid )
test_labels = np.array(label_test)


## Training the model with SimpleRNN

In [ ]:
embedding_vector_length = 32
model1 = Sequential()
model1.add(Embedding(vocab_size, embedding_vector_length, input_length=max_length))
model1.add(SimpleRNN(100))  # Use SimpleRNN here.
model1.add(Dense(1, activation='sigmoid'))

model1.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model1.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 150, 32)           320000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 100)               13300     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 333401 (1.27 MB)
Trainable params: 333401 (1.27 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model1.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 26s 74ms/step - loss: 0.6966 - accuracy: 0.4995 - val_loss: 0.7120 - val_accuracy: 0.4910
Epoch 2/5
313/313 [==============================] - 22s 70ms/step - loss: 0.6969 - accuracy: 0.5035 - val_loss: 0.6968 - val_accuracy: 0.4934
Epoch 3/5
313/313 [==============================] - 20s 65ms/step - loss: 0.6939 - accuracy: 0.5117 - val_loss: 0.7030 - val_accuracy: 0.4876
Epoch 4/5
313/313 [==============================] - 22s 71ms/step - loss: 0.6917 - accuracy: 0.5290 - val_loss: 0.6962 - val_accuracy: 0.4996
Epoch 5/5
313/313 [==============================] - 21s 66ms/step - loss: 0.6842 - accuracy: 0.5508 - val_loss: 0.6970 - val_accuracy: 0.5128


In [ ]:
# Final evaluation of the model.
scores = model1.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 50.75%


## Training the model with LSTM

## Training the LSTM model
Now, we train our model with the training data.

In [ ]:
# Create the model.
embedding_vector_length = 32
model2 = Sequential()
model2.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model2.add(LSTM(100))
model2.add(Dense(1, activation='sigmoid'))
model2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model2.summary())


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 150, 32)           320000    
                                                                 
 lstm (LSTM)                 (None, 100)               53200     
                                                                 
 dense_1 (Dense)             (None, 1)                 101       
                                                                 
Total params: 373301 (1.42 MB)
Trainable params: 373301 (1.42 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model2.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 71s 218ms/step - loss: 0.5823 - accuracy: 0.6665 - val_loss: 0.4652 - val_accuracy: 0.8176
Epoch 2/5
313/313 [==============================] - 68s 217ms/step - loss: 0.3291 - accuracy: 0.8726 - val_loss: 0.3720 - val_accuracy: 0.8492
Epoch 3/5
313/313 [==============================] - 70s 223ms/step - loss: 0.2511 - accuracy: 0.9075 - val_loss: 0.3522 - val_accuracy: 0.8444
Epoch 4/5
313/313 [==============================] - 68s 216ms/step - loss: 0.2229 - accuracy: 0.9247 - val_loss: 0.3878 - val_accuracy: 0.8474
Epoch 5/5
313/313 [==============================] - 70s 222ms/step - loss: 0.1746 - accuracy: 0.9427 - val_loss: 0.4370 - val_accuracy: 0.8428


In [ ]:
# Final evaluation of the model.
scores = model2.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 82.65%


## Training the model with GRU

In [ ]:
# create the model.
embedding_vector_length = 32
model3 = Sequential()
model3.add(Embedding(vocab_size, output_dim = embedding_vector_length, input_length=max_length))
model3.add(GRU(100))  # Use GRU instead of LSTM.
model3.add(Dense(1, activation='sigmoid'))
model3.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model3.summary())

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 150, 32)           320000    
                                                                 
 gru (GRU)                   (None, 100)               40200     
                                                                 
 dense_2 (Dense)             (None, 1)                 101       
                                                                 
Total params: 360301 (1.37 MB)
Trainable params: 360301 (1.37 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
model3.fit(train_padded, train_labels, validation_data=(valid_padded, valid_labels), epochs=5, batch_size=64)

Epoch 1/5
313/313 [==============================] - 65s 201ms/step - loss: 0.6757 - accuracy: 0.5540 - val_loss: 0.6259 - val_accuracy: 0.6372
Epoch 2/5
313/313 [==============================] - 69s 219ms/step - loss: 0.6057 - accuracy: 0.6784 - val_loss: 0.5890 - val_accuracy: 0.6782
Epoch 3/5
313/313 [==============================] - 65s 208ms/step - loss: 0.4744 - accuracy: 0.7790 - val_loss: 0.4462 - val_accuracy: 0.8146
Epoch 4/5
313/313 [==============================] - 66s 210ms/step - loss: 0.3187 - accuracy: 0.8741 - val_loss: 0.3783 - val_accuracy: 0.8424
Epoch 5/5
313/313 [==============================] - 69s 220ms/step - loss: 0.2414 - accuracy: 0.9128 - val_loss: 0.3926 - val_accuracy: 0.8420


In [ ]:
# Final evaluation of the model.
scores = model3.evaluate(test_padded, test_labels, verbose=0)
print("Accuracy: %.2f%%" % (scores[1]*100))

Accuracy: 82.34%


## Key information
This demonstration illustrated how to perform sentiment classification using advanced RNNs.

## Reflect
What are the practical applications of these techniques?
> Select the pen from the toolbar to add your entry.